Read the Product File

In [0]:
df=spark.read.csv("/Volumes/productcatalog/productschema/filestorage/csv/products-10000.csv")
df.display() ## CSV with display gives column c1, c2

In [0]:
df=spark.read.csv("/Volumes/productcatalog/productschema/filestorage/csv/products-10000.csv",header=True)
df.display() 

In [0]:
df=spark.read.csv("/Volumes/productcatalog/productschema/filestorage/csv/products-10000.csv",header=True,inferSchema=True)
df.display()

In [0]:
df.display(10) ## Arg given is 10 but displays all records

In [0]:
df.show(10) ## shows 10 only but html format

In [0]:
##df.limit(10).display() ## 
display(df.limit(10))  ##  Same

**Dynamic Syntax for READING DATA**

In [0]:
df_csv=spark.read.format("csv")\
    .option("header",True)\
        .option("inferSchema",True)\
            .load("/Volumes/productcatalog/productschema/filestorage/csv/products-10000.csv")
df_csv.display()


In [0]:
df_csv=spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load("/Volumes/productcatalog/productschema/filestorage/csv/products-10000.csv")
display(df_csv)

**Reading Parequet files**

In [0]:
df_Parquet= spark.read.format("parquet").option("header",True).option("inferSchema",True).load("/Volumes/productcatalog/productschema/filestorage/parquet/titanic.parquet")
df_Parquet.display()

> **CREATE TABLE in PRODUCT CATALOG --> BRONZE SCHEMA**

In [0]:
filepath="/Volumes/productcatalog/productschema/filestorage/csv/products-10000.csv"
df=spark.read.format("csv").option("header","true").option("inferSchema","true")\
    .load(filepath).withColumnRenamed("Internal ID","InternalID")
df.display()

df.write.mode("overwrite").saveAsTable("productcatalog.bronze.Product_bronze")

*Read Managed Table from Catalog*

In [0]:
%sql
select * from productcatalog.bronze.product_bronze

In [0]:
spark.read.table("productcatalog.bronze.Product_bronze").display()


In [0]:
spark.sql("select * from productcatalog.bronze.Product_bronze").display()

**Ingesting Data to SILVER**

In [0]:
%sql
Select *
, CASE WHEN SIZE IN ('S','Small') Then 'S'
 WHEN Size in ('M','Medium') Then 'M'
 WHEN Size in ('L','Large') Then 'L'
 WHEN Size in ('XL','Extra Large') Then 'XL'
 WHEN Size in ('XXL','Extra Extra Large') Then 'XXL'
 ELSE 'Unkown' END as SizeNew
 from productcatalog.bronze.product_bronze

In [0]:
df=spark.sql("""Select *
, CASE WHEN SIZE IN ('S','Small') Then 'S'
 WHEN Size in ('M','Medium') Then 'M'
 WHEN Size in ('L','Large') Then 'L'
 WHEN Size in ('XL','Extra Large') Then 'XL'
 WHEN Size in ('XXL','Extra Extra Large') Then 'XXL'
 ELSE 'Unkown' END as SizeNew
 from productcatalog.bronze.product_bronze""")

df.display()

In [0]:
df=spark.sql("""Select *
, CASE WHEN SIZE IN ('S','Small') Then 'S'
 WHEN Size in ('M','Medium') Then 'M'
 WHEN Size in ('L','Large') Then 'L'
 WHEN Size in ('XL','Extra Large') Then 'XL'
 WHEN Size in ('XXL','Extra Extra Large') Then 'XXL'
 ELSE 'Unkown' END as SizeNew
 from productcatalog.bronze.product_bronze""")

df.write.mode("overwrite").saveAsTable("productcatalog.silver.product_silver")

In [0]:
spark.read.table("productcatalog.silver.product_silver").limit(5).display()

**Ingesting Data from Silver to GOLD**

In [0]:
df_silver=spark.read.table("productcatalog.silver.product_silver")
df_silver.limit(5).display()

df_silver_selected=df_silver.select("Name","Description","Brand","Category","Price","Stock","SizeNew","Availability")
df_silver_selected.limit(10).display()

In [0]:
%sql
select SizeNew, count(*) as Size_CNT
 from productcatalog.silver.product_silver
group by SizeNew
order by SizeNew;

select category, count(*) as Category_CNT
 from productcatalog.silver.product_silver
group by category order by Category
;

In [0]:
df_gold_size=spark.sql("""select SizeNew, count(*) as Size_CNT
 from productcatalog.silver.product_silver
group by SizeNew
order by SizeNew""")
df_gold_size.display()

df_gold_category=spark.sql("""select category, count(*) as Category_CNT
 from productcatalog.silver.product_silver
group by category order by Category""")
df_gold_category.display()

In [0]:
df_gold_size.write.mode("overwrite").saveAsTable("productcatalog.gold.size_gold")
df_gold_category.write.mode("overwrite").saveAsTable("productcatalog.gold.category_gold")


In [0]:
spark.read.table("productcatalog.gold.size_gold").display()
spark.read.table("productcatalog.gold.category_gold").display()

In [0]:
spark.sql("select * from productcatalog.gold.size_gold").display()
spark.sql("select * from productcatalog.gold.category_gold").display()

In [0]:
%sql
select * from productcatalog.gold.size_gold;
select * from productcatalog.gold.category_gold;